In [6]:
import pandas as pd
import numpy as np


### Inspect dataset

In [7]:
df= pd.read_csv("/content/customer_ml_feature.csv")

In [8]:
print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully
Rows: 96096
Columns: 20


In [9]:
df.head()

,customer_unique_id,customer_id_count,customer_city,customer_state,total_orders,delivered_orders,canceled_orders,unavailable_orders,delivered_item_revenue,delivered_freight,delivered_item_count,average_delivered_order_value,first_purchase_date,last_purchase_date,recency_days,average_delivery_days,average_review_score,reviewed_orders,repeat_customer,feature_reference_date
0,0000f46a3911fa3c0805444483337064,1,sao jose,SC,1,1,0,0,69.00,17.22,1,69.00,2017-03-10 21:05:03,2017-03-10 21:05:03,587,25.731759,3.0,1,0,2018-10-18
1,0000f6ccb0745a6a4b88665a16c9f078,1,belem,PA,1,1,0,0,25.99,17.63,1,25.99,2017-10-12 20:29:41,2017-10-12 20:29:41,371,20.037083,4.0,1,0,2018-10-18
2,00050ab1314c0e55a6ca13cf7181fecf,1,campinas,SP,1,1,0,0,27.99,7.39,1,27.99,2018-04-20 12:57:23,2018-04-20 12:57:23,181,6.966389,4.0,1,0,2018-10-18
3,00053a61a98854899e70ed204dd4bafe,1,curitiba,PR,1,1,0,0,382.00,37.18,2,382.00,2018-02-28 11:15:41,2018-02-28 11:15:41,232,16.324421,1.0,1,0,2018-10-18
4,0006fdc98a402fceb4eb0ee528f6a8d4,1,mimoso do sul,ES,1,1,0,0,13.90,15.10,1,13.90,2017-07-18 09:23:10,2017-07-18 09:23:10,457,16.388646,3.0,1,0,2018-10-18


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   customer_unique_id             96096 non-null  object 
 1   customer_id_count              96096 non-null  int64  
 2   customer_city                  96096 non-null  object 
 3   customer_state                 96096 non-null  object 
 4   total_orders                   96096 non-null  int64  
 5   delivered_orders               96096 non-null  int64  
 6   canceled_orders                96096 non-null  int64  
 7   unavailable_orders             96096 non-null  int64  
 8   delivered_item_revenue         96096 non-null  float64
 9   delivered_freight              96096 non-null  float64
 10  delivered_item_count           96096 non-null  int64  
 11  average_delivered_order_value  93358 non-null  float64
 12  first_purchase_date            96096 non-null 

#### Duplicate and missing checks

In [11]:
print("Duplicate customer IDs:",
      df["customer_unique_id"].duplicated().sum())

print("Duplicate complete rows:",
      df.duplicated().sum())

print("\nMissing values by column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Duplicate customer IDs: 0
Duplicate complete rows: 0

Missing values by column:
average_delivered_order_value    2738
average_delivery_days            2746
average_review_score              716
dtype: int64


In [12]:
print("Customers without delivered orders:",
      (df["delivered_orders"] == 0).sum())

print("Customers with missing average order value:",
      df["average_delivered_order_value"].isna().sum())

print("Customers without reviewed orders:",
      (df["reviewed_orders"] == 0).sum())

print("Customers with missing review score:",
      df["average_review_score"].isna().sum())

print("\nDelivery days by delivered order count:")
print(
    df.groupby("delivered_orders")["average_delivery_days"]
      .agg(["count", "size"])
      .head(10)
)

Customers without delivered orders: 2738
Customers with missing average order value: 2738
Customers without reviewed orders: 716
Customers with missing review score: 716

Delivery days by delivered order count:
                  count   size
delivered_orders              
0                     0   2738
1                 90549  90557
2                  2573   2573
3                   181    181
4                    28     28
5                     9      9
6                     5      5
7                     3      3
9                     1      1
15                    1      1


In [19]:
# Customer feature dictionary preparation

feature_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
})

print(feature_info.to_string(index=False))

                       Column      Data Type  Missing Values  Unique Values
           customer_unique_id         object               0          96096
            customer_id_count          int64               0              9
                customer_city         object               0           4118
               customer_state         object               0             27
                 total_orders          int64               0              9
             delivered_orders          int64               0             10
              canceled_orders          int64               0              4
           unavailable_orders          int64               0              3
       delivered_item_revenue        float64               0           8144
            delivered_freight        float64               0           7811
         delivered_item_count          int64               0             20
average_delivered_order_value        float64            2738           8502
          fi

### Convert dates


In [13]:
date_columns = [
    "first_purchase_date",
    "last_purchase_date",
    "feature_reference_date"
]

for column in date_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

print(df[date_columns].dtypes)

first_purchase_date       datetime64[ns]
last_purchase_date        datetime64[ns]
feature_reference_date    datetime64[ns]
dtype: object


In [14]:
print("First purchase after last purchase:",
      (df["first_purchase_date"] > df["last_purchase_date"]).sum())

print("Last purchase after reference date:",
      (df["last_purchase_date"] > df["feature_reference_date"]).sum())

print("Negative recency:",
      (df["recency_days"] < 0).sum())

First purchase after last purchase: 0
Last purchase after reference date: 0
Negative recency: 0


### Date consistency

In [15]:

missing_delivery = df[
    (df["delivered_orders"] > 0) &
    (df["average_delivery_days"].isna())
]

print("Customers with delivered orders but missing delivery time:",
      len(missing_delivery))

print(
    missing_delivery[
        [
            "customer_unique_id",
            "total_orders",
            "delivered_orders",
            "first_purchase_date",
            "last_purchase_date",
            "average_delivery_days"
        ]
    ].head(10)
)


Customers with delivered orders but missing delivery time: 8
                     customer_unique_id  total_orders  delivered_orders  \
20318  d77cf4be2654aa70ef150f8bfec076a6             1                 1   
26248  175378436e2978be55b8f4316bce4811             1                 1   
28556  2f17c5b324ad603491521b279a9ff4de             1                 1   
29703  3bc508d482a402715be4d5cf4020cc81             1                 1   
46257  ebf7e0d43a78c81991a4c59c145c75db             1                 1   
50006  13467e882eb3a701826435ee4424f2bd             1                 1   
74725  1bd06a0c0df8b23dacfd3725d2dc0bb9             1                 1   
91307  cce5e8188bf42ffb3bb5b18ff58f5965             1                 1   

      first_purchase_date  last_purchase_date  average_delivery_days  
20318 2017-05-25 23:22:43 2017-05-25 23:22:43                    NaN  
26248 2018-06-27 16:09:12 2018-06-27 16:09:12                    NaN  
28556 2018-06-20 06:58:43 2018-06-20 06:58:43     

##### Business rule validation

In [16]:

checks = {
    "Total orders at least 1":
        (df["total_orders"] >= 1).all(),

    "Delivered orders do not exceed total orders":
        (df["delivered_orders"] <= df["total_orders"]).all(),

    "Canceled orders do not exceed total orders":
        (df["canceled_orders"] <= df["total_orders"]).all(),

    "Unavailable orders do not exceed total orders":
        (df["unavailable_orders"] <= df["total_orders"]).all(),

    "No negative delivered item counts":
        (df["delivered_item_count"] >= 0).all(),

    "No negative revenue":
        (
            (df["delivered_item_revenue"] >= 0) &
            (df["delivered_freight"] >= 0)
        ).all(),

    "Review scores between 1 and 5":
        df["average_review_score"].dropna().between(1, 5).all(),

    "Repeat customer flag is correct":
        (
            df["repeat_customer"] ==
            (df["total_orders"] > 1).astype(int)
        ).all()
}

for check, result in checks.items():
    print(("PASS" if result else "FAIL"), check)


PASS Total orders at least 1
PASS Delivered orders do not exceed total orders
PASS Canceled orders do not exceed total orders
PASS Unavailable orders do not exceed total orders
PASS No negative delivered item counts
PASS No negative revenue
PASS Review scores between 1 and 5
PASS Repeat customer flag is correct


#### 	Financial reconciliation

In [17]:

summary = {
    "Total customers": len(df),
    "Total orders": df["total_orders"].sum(),
    "Delivered orders": df["delivered_orders"].sum(),
    "Delivered product revenue": round(
        df["delivered_item_revenue"].sum(), 2
    ),
    "Delivered freight": round(
        df["delivered_freight"].sum(), 2
    ),
    "Total delivered revenue": round(
        df["delivered_item_revenue"].sum()
        + df["delivered_freight"].sum(), 2
    ),
    "Repeat customers": df["repeat_customer"].sum()
}

for metric, value in summary.items():
    print(f"{metric}: {value}")


Total customers: 96096
Total orders: 99441
Delivered orders: 96478
Delivered product revenue: 13221498.11
Delivered freight: 2198275.64
Total delivered revenue: 15419773.75
Repeat customers: 2997


#### Create the final validation report

In [18]:

# Final customer feature validation report

checks = {
    "Correct customer row count": len(df) == 96096,

    "No duplicate customer IDs":
        df["customer_unique_id"].duplicated().sum() == 0,

    "No duplicate complete rows":
        df.duplicated().sum() == 0,

    "Missing average order value explained":
        df["average_delivered_order_value"].isna().sum()
        == (df["delivered_orders"] == 0).sum(),

    "Missing review scores explained":
        df["average_review_score"].isna().sum()
        == (df["reviewed_orders"] == 0).sum(),

    "Missing delivery values accounted for":
        df["average_delivery_days"].isna().sum() == 2746,

    "No invalid purchase dates":
        (df["first_purchase_date"] <=
         df["last_purchase_date"]).all(),

    "No future purchase dates":
        (df["last_purchase_date"] <=
         df["feature_reference_date"]).all(),

    "No negative recency":
        (df["recency_days"] >= 0).all(),

    "Order totals reconcile":
        df["total_orders"].sum() == 99441,

    "Delivered orders reconcile":
        df["delivered_orders"].sum() == 96478,

    "Delivered product revenue reconciles":
        round(df["delivered_item_revenue"].sum(), 2)
        == 13221498.11,

    "Delivered freight reconciles":
        round(df["delivered_freight"].sum(), 2)
        == 2198275.64,

    "Repeat customers reconcile":
        df["repeat_customer"].sum() == 2997
}

report = pd.DataFrame({
    "Validation": checks.keys(),
    "Result": [
        "PASS" if result else "FAIL"
        for result in checks.values()
    ]
})

print(report.to_string(index=False))

passed = sum(checks.values())
failed = len(checks) - passed

print("\nTotal checks:", len(checks))
print("Passed:", passed)
print("Failed:", failed)

if failed == 0:
    print("\nFINAL STATUS: PASSED WITH DOCUMENTED DATA LIMITATIONS")
else:
    print("\nFINAL STATUS: REVIEW REQUIRED")


                           Validation Result
           Correct customer row count   PASS
            No duplicate customer IDs   PASS
           No duplicate complete rows   PASS
Missing average order value explained   PASS
      Missing review scores explained   PASS
Missing delivery values accounted for   PASS
            No invalid purchase dates   PASS
             No future purchase dates   PASS
                  No negative recency   PASS
               Order totals reconcile   PASS
           Delivered orders reconcile   PASS
 Delivered product revenue reconciles   PASS
         Delivered freight reconciles   PASS
           Repeat customers reconcile   PASS

Total checks: 14
Passed: 14
Failed: 0

FINAL STATUS: PASSED WITH DOCUMENTED DATA LIMITATIONS



## 10. Conclusion

The customer feature dataset was successfully validated using
Python and Pandas against the PostgreSQL source data.

The final dataset contains 96,096 unique customers and 20 columns.

### Key validation results

1. No duplicate customer records were identified.
2. Customer order counts matched PostgreSQL.
3. Delivered order counts matched PostgreSQL.
4. Product revenue and freight reconciled exactly.
5. Customer purchase dates and recency values were consistent.
6. Business rule checks passed.
7. Missing average order values and review scores were
   explained by unavailable customer activity.

### Data limitations


1. Average delivered order value is missing for 2,738 customers
   because they have no delivered orders.

2. Average review score is missing for 716 customers
   because they have no reviewed orders.

3. Average delivery duration is missing for 2,746 customers.
   Of these, 2,738 have no delivered orders, while 8 have
   delivered orders with missing delivery timestamps.

Missing values were retained as null rather than replaced
with zero or estimated values.

### Final outcome

The customer feature dataset passed the defined SQL
and Python validation checks, with the documented
source data limitations.

The dataset is suitable for further exploratory analysis
and feature preparation. Predictive modeling requires
a separately defined prediction date and target variable,
with appropriate prevention of data leakage.
